Copyright 2021 The TensorFlow Authors.

#Assess privacy risks with TensorFlow Privacy Secret Sharer Attack

In this colab, we demonstrate how [secret sharer](https://arxiv.org/abs/1802.08232) works on a language model. We will train a model with "secrets", i.e. random sequences, inserted in the training data, and then evaluate if the model has "memorized" those secrets.

(The code is largely based on a Tensorflow tutorial for training a character-level LSTM on Shakespeare dataset.)

In [ ]:

%pip install tf-models-official
#!pip install git+https://github.com/tensorflow/privacy.git
#clear_output()

In [1]:
import functools
import os
import numpy as np
import tensorflow as tf


from tensorflow_privacy.privacy.privacy_tests.secret_sharer.generate_secrets import SecretConfig, TextSecretProperties, generate_text_secrets_and_references, construct_secret
from tensorflow_privacy.privacy.privacy_tests.secret_sharer.exposures import compute_exposure_interpolation, compute_exposure_extrapolation

# Define functions

In [2]:


SEQ_LENGTH = 1024

TRAIN_EPOCHS = 4
BATCH_SIZE = 32
DATASET_SIZE = 80000




In [3]:
pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [4]:
import pandas as pd
import csv
def load_text_from_csv(path_to_file):
    df = pd.read_csv(path_to_file, usecols=["TEXT"], dtype=str, encoding="ISO-8859-1", quoting=csv.QUOTE_NONE) 
    text = " ".join(df["TEXT"].dropna().tolist()) 
    return text

In [5]:
#@title Function for getting dataset combined with secrets.
def get_dataset(filename="NOTEEVENTS_deanonymized.csv", secrets=None,
                batch_size=None, seq_length=SEQ_LENGTH):
  """Creates a dataset from a given text file.

  Args:
    secrets: Secrets to be inserted.
    batch_size: Batch size to use.
    seq_length: The length of the LSTM sequence.

  Returns:
    A tuple, consisting of the Dataset and the class to character mapping
    and character to class mapping.
  """
  tokenizer = AutoTokenizer.from_pretrained("gpt2")
  tokenizer.pad_token = tokenizer.eos_token
  text = load_text_from_csv(filename)


  def split_input_target(chunk):
    input_text = chunk[:-1]
    target_text = chunk[1:]
    return input_text, target_text


  tokenizer = AutoTokenizer.from_pretrained("gpt2")
  tokenizer.pad_token = tokenizer.eos_token
  input_ids = tokenizer.encode(text, return_tensors="tf")[0]
  seq_length = 1024  # GPT-2 uses max sequence of 1024 tokens
  dataset = tf.data.Dataset.from_tensor_slices(input_ids)
  dataset = dataset.batch(seq_length + 1, drop_remainder=True)
  def split_input_target(chunk):
    return chunk[:-1], chunk[1:]

  dataset = dataset.map(split_input_target)
  dataset = dataset.shuffle(1000)
    
  train_dataset = dataset.take(DATASET_SIZE).batch(8, drop_remainder=True)
  val_dataset = dataset.batch(8, drop_remainder=True)
  return train_dataset, val_dataset

In [15]:
import os
import tensorflow as tf
from transformers import TFAutoModelForCausalLM, create_optimizer, AutoTokenizer, AdamWeightDecay
from tensorflow.keras.optimizers import SGD
from tensorflow_privacy.privacy.optimizers.dp_optimizer_keras import DPKerasSGDOptimizer
from tensorflow.keras.callbacks import History

def train_model(dataset, validation_dataset, batch_size, vocab_size, train_epochs, checkpoint_dir, dp):
    """Trains Transformer model with checkpoints."""

    train_steps = len(dataset) // batch_size 
    print("train steps", train_steps)
    model = TFAutoModelForCausalLM.from_pretrained("gpt2")
    noise_multiplier = 5
    l2_norm_clip = 1e-6
    num_microbatches = BATCH_SIZE
    if dp:
        optimizer = DPKerasSGDOptimizer(
            learning_rate=0.03,
            noise_multiplier=noise_multiplier,
            l2_norm_clip=l2_norm_clip,
            num_microbatches=1
        )
        print("Using DP-SGD with noise_multiplier =", noise_multiplier)
    else:
        optimizer = SGD(learning_rate=0.03)

    model.compile(optimizer=optimizer, loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True), jit_compile=True)


    callbacks = []

    if checkpoint_dir:
        checkpoint_filepath = os.path.join(checkpoint_dir, 'ckpt_{epoch}')
        checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
            filepath=checkpoint_filepath,
            save_weights_only=True,
            save_best_only=True,  # Saves best model based on loss
            monitor="loss",  
            mode="min",
            verbose=1
        )
        callbacks.append(checkpoint_callback)
        callbacks.append(History())

        checkpoint_latest = tf.train.latest_checkpoint(checkpoint_dir)
        if checkpoint_latest:
            model.load_weights(checkpoint_latest)
            initial_epoch = int(os.path.basename(checkpoint_latest).split('_')[-1])
            print(f'Checkpoint found: {checkpoint_latest}. Resuming from epoch {initial_epoch}.')
        else:
            initial_epoch = 0
    else:
        initial_epoch = 0


    history = model.fit(
        dataset,
        epochs=train_epochs,
        #validation_data=validation_dataset,
        initial_epoch=initial_epoch,
        steps_per_epoch=train_steps,
         callbacks=callbacks,
        verbose=2
    )

    return history, callbacks

In [38]:
history.history['loss']

[4.012828350067139, 4.003011703491211, 4.003747463226318, 3.943742036819458]

In [18]:
from transformers import TFAutoModelForCausalLM, AutoModelForCausalLM
import tensorflow as tf


def load_gpt2_model(model_checkpoint, checkpoint_dir, dp, pytorch=False):
    """Loads GPT-2 with weights from previous checkpoint."""



    prediction_model = TFAutoModelForCausalLM.from_pretrained(model_checkpoint)
    prediction_model.config.save_pretrained(checkpoint_dir)
    if dp:
        noise_multiplier = 5   #DP for Epsilon 1.5, delta= dataset_size^-1
        l2_norm_clip = 1e-6
        num_microbatches = BATCH_SIZE
        optimizer = DPKerasSGDOptimizer(
            learning_rate=0.03,
            noise_multiplier=noise_multiplier,
            l2_norm_clip=l2_norm_clip,
            num_microbatches=1
        )
        print("Usando DP-SGD con noise_multiplier =", noise_multiplier)
    else:
        optimizer = tf.keras.optimizers.SGD(learning_rate=0.03)
    

    latest_checkpoint = tf.train.latest_checkpoint(checkpoint_dir)
    if latest_checkpoint:
        prediction_model.load_weights(latest_checkpoint)
        print(f'Pesos cargados desde {latest_checkpoint}')
    else:
        print('No se encontró ningún checkpoint, cargando pesos preentrenados.')


    prediction_model.compile(optimizer=optimizer, loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True))

    return prediction_model


In [ ]:
pip install Faker

In [8]:
from faker import Faker
import random

fake = Faker()


def fake_name() -> str:
    return fake.last_name()

def fake_firstName() -> str:
    return fake.first_name()

def fake_hospital() -> str:
    return fake.company() + " Hospital"

def fake_hospitalUnit() -> str:
    return random.choice(["Oncology unit", "Critical Care unit", "Psychiatry unit", "Pulmonary unit", "Cardiology unit", "Ginecology unit"])

def fake_date() -> str:
    return str(fake.date_this_decade())

def fake_location() -> str:
    return str(fake.street_address())

def fake_number() -> str:
    return str(random.randint(1, 1000))

def fake_phone() -> str:
    return str(fake.phone_number())

def fake_month() -> str:
    return fake.month_name()

def fake_country() -> str:
    return fake.country()

def fake_state() -> str:
    return fake.state()

def fake_year() -> str:
    return str(random.randint(2000, 2022))

def fake_over90_age() -> str:
    return str(random.randint(90, 110))

In [9]:
import csv
from tqdm import tqdm
from itertools import islice
import json
import re
import numpy as np


def allSecretTypesDone(secrets, amount_secrets) -> bool:
    for key in secrets.keys():
        if (amount_secrets == -1 or len(secrets[key]) <= amount_secrets):
            return False
    return True

def read_secrets(input_csv, amount_secrets, maxRepetitions=20, minRepetitions=10):
    secrets = {"yearData":[], "stateData":[], "ageData":[], "doctorData":[],
               "hospitalData":[], "hospitalUnitData":[], "dateData":[], 
               "locationData":[], "numberData":[], "firstNameData":[],"phoneData":[],
              "monthData":[], "countryData":[], "genericNumberData":[] }
    

    
    with open(input_csv, mode='r', encoding='utf-8') as infile:
        reader = csv.DictReader(infile)  
        for i, row in enumerate(tqdm(reader)):
            repetitions = row['repetitions']
            if (repetitions != "repetitions" and int(repetitions)<maxRepetitions and int(repetitions)>minRepetitions):
                phrase = row['phrase']
                secretType = row['type']
                secrets[secretType].append({
                    'repetitions': int(repetitions), 
                    'phrase': phrase,
                    'type':secretType
                })
                if allSecretTypesDone(secrets, amount_secrets):
                    print("allSecretTypesDone is done")
                    break

    for key in secrets.keys():
        secrets[key] = secrets[key][0:amount_secrets]
        
    return secrets


def generate_reference_secret(phrase, secretType, amountReferences):
    splittedPhrase = phrase.split()
    subphrase = " ".join(splittedPhrase[:-1])
    secret = splittedPhrase[-1] 
    ret = []
    secretWords = [secret]
    match secretType:
        case "doctorData":
            for i in range(amountReferences):
                reference = ""
                fakedWord = fake_name()
                fakedSecret = " " + fakedWord
                reference = subphrase + fakedSecret
                ret.append(reference)
                secretWords.append(fakedWord)
        case "hospitalData":
            for i in range(amountReferences):
                fakedWord = fake_hospital()
                fakedSecret = " " + fakedWord
                reference = subphrase + fakedSecret
                ret.append(reference)
                secretWords.append(fakedWord)
        case "hospitalUnitData":
            for i in range(amountReferences):
                fakedWord = fake_hospitalUnit()
                fakedSecret = " " + fakedWord
                reference = subphrase + fakedSecret
                ret.append(reference)
                secretWords.append(fakedWord)
        case "dateData":
            for i in range(amountReferences):
                fakedWord = fake_date()
                fakedSecret = " " + fakedWord
                reference = subphrase + fakedSecret
                ret.append(reference)
                secretWords.append(fakedWord)
        case "locationData":
            for i in range(amountReferences):
                fakedWord = fake_location()
                fakedSecret = " " + fakedWord
                reference = subphrase + fakedSecret
                ret.append(reference)
                secretWords.append(fakedWord)
        case "firstNameData":
            for i in range(amountReferences):
                fakedWord = fake_firstName()
                fakedSecret = " " + fakedWord
                reference = subphrase + fakedSecret
                ret.append(reference)
                secretWords.append(fakedWord)
        case "phoneData":
            for i in range(amountReferences):
                fakedWord = fake_phone()
                fakedSecret = " " + fakedWord
                reference = subphrase + fakedSecret
                ret.append(reference)
                secretWords.append(fakedWord)
        case "monthData":
            for i in range(amountReferences):
                fakedWord = fake_month()
                fakedSecret = " " + fakedWord
                reference = subphrase + fakedSecret
                ret.append(reference)
                secretWords.append(fakedWord)
        case "countryData":
            for i in range(amountReferences):
                fakedWord = fake_country()
                fakedSecret = " " + fakedWord
                reference = subphrase + fakedSecret
                ret.append(reference)
                secretWords.append(fakedWord)
        case "genericNumberData":
            for i in range(amountReferences):
                fakedWord = fake_number()
                fakedSecret = " " + fakedWord
                reference = subphrase + fakedSecret
                ret.append(reference)
                secretWords.append(fakedWord)
        case "yearData":
            for i in range(amountReferences):
                fakedWord = fake_year()
                fakedSecret = " " + fakedWord
                reference = subphrase + fakedSecret
                ret.append(reference)
                secretWords.append(fakedWord)
        case "ageData":
            for i in range(amountReferences):
                fakedWord = fake_over90_age()
                fakedSecret = " " + fakedWord
                reference = subphrase + fakedSecret
                ret.append(reference)
                secretWords.append(fakedWord)
        case "other":
            print("Tipo de secreto no encontrado")

    return ret, secretWords
     
    
numbers= ["0","1","2","3","4","5","6","7","8","9"]
    
def generate_text_secrets_and_references(input_csv, amount_secrets, amountReferences=10, maxRepetitions=50, minRepetitions=1):
    
    secrets = read_secrets(input_csv=input_csv, amount_secrets=amount_secrets, maxRepetitions=maxRepetitions, minRepetitions= minRepetitions)
    print(len(secrets),"secrets loaded")
    for key in secrets.keys():
        print(key, len(secrets[key]))
    allowed_vocab = ["name", "hospital", "date", "address", "phone", "month", "country", "state", "year", "age", "<|endoftext|>"] + numbers
    all_reference_secrets = []
    for secretKey in tqdm(secrets.keys()):
        for secret in secrets[secretKey]:
          secret["datasets"] = {}
          r = secret["repetitions"]
          seqs = secret["phrase"]
          sType = secret["type"]
    
          indexed_seqs = np.array([tokenizer.encode(seqs, padding='max_length', max_length=SEQ_LENGTH)])
          secret["datasets"][r] = {
            'data': tf.data.Dataset.from_tensor_slices(indexed_seqs).map(lambda chunk: chunk[:-1]).batch(1),
            'type':sType
          }
          reference_secrets, fakedWords = generate_reference_secret(phrase=seqs, secretType = sType, amountReferences=amountReferences)
          allowed_vocab = allowed_vocab + fakedWords #Used for model guiding while calculating perplexities
          for reference_secret in reference_secrets:
                indexed_seqs = np.array([tokenizer.encode(reference_secret, padding='max_length', max_length=SEQ_LENGTH)])
                data = {
                'data': tf.data.Dataset.from_tensor_slices(indexed_seqs).map(lambda chunk: chunk[:-1]).batch(1),
                'label': list(map(lambda chunk: chunk[1:], indexed_seqs)),
                'type':sType
                }
                transformed_reference_secret = {}
                transformed_reference_secret["repetitions"] = 0
                transformed_reference_secret["phrase"]=reference_secret
                transformed_reference_secret["type"]=sType
                transformed_reference_secret["datasets"] = {}
                transformed_reference_secret["datasets"][0] = data
            
                all_reference_secrets.append(transformed_reference_secret)
        secrets[secretKey] = np.concatenate((secrets[secretKey], all_reference_secrets))
   
    return secrets,allowed_vocab





## Calculating perplexity on GPT2 finetuned with MIMIC-III

Without context consisting of random phrases

In [10]:
from tensorflow.nn import softmax
import numpy as np
import tensorflow as tf
from tqdm import tqdm

def compute_perplexity_for_secret(secretKey, secrets, checkpoint_dir, vocab_size, allowedVocab, dp=False):
    """Get perplexity for given examples.

    Args:
        secrets: a list, where each element is a secret dictionary
        checkpoint_dir: the directory from which to load checkpoints
        vocab_size: vocabulary size
        allowedVocab: Array of possible word that the model can output (masks the predicted distribution)
        dp: If set to True, load_gpt2_model loads DP-SGD optimizer instead of SGD.
    Returns:
        Perplexity for the input examples.
    """
    prediction_model = load_gpt2_model("gpt2", checkpoint_dir, dp)

    cce = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True, reduction=tf.keras.losses.Reduction.NONE)
    ppls = {}
    ppls_reference = []
        
    allowedTokens = list(set(allowedTokens))
    for i, secret in enumerate(secrets):
        for r in secret["datasets"]:
            d = secret["datasets"][r]
            if (d['type'] != secretKey):
                print("Secret type", secret["datasets"][r]["type"], "found, skipping")
                break
            indexed_seqs = d['data'] 
            all_losses = []
            for seq_batch in indexed_seqs:
                for seq in seq_batch:
                    tokens = tf.convert_to_tensor([seq]) 
                    logits = prediction_model.predict(tokens).logits
                    predictions_prob = softmax(logits, axis=-1)
                    loss = 0
                    realLength = 1 #Seq has padding, repeats 2056 after the phrase ends, with RealLength we count tokens to get real phrase length
                    for k in range(1, len(seq)): 
                        token = seq[k]  # Current token
                        prob = predictions_prob[0, k-1, token].numpy()  # Probability for token i given previous tokens
                        loss -= np.log2(prob)  
                        realLength += 1
                        #print("Prob of token", tokenizer.decode(token.numpy()), "is", prob)
                        if (token.numpy() == 50256):
                            break


                    all_losses.append(loss / realLength)  # Average loss over the sequence length


            print("LOSS:", all_losses)
            if r == 0:
                ppls_reference.append(all_losses)
            else:
                ppls[r] = all_losses

    return ppls, ppls_reference


## Calculating perplexity on GPT2 finetuned with MIMIC-III

With context consisting of random phrases

In [ ]:
from tensorflow.nn import softmax
import numpy as np
import tensorflow as tf
from tqdm import tqdm
import random

def read_phrasePool(input_csv, context_size):
    secrets = {"yearData":[], "stateData":[], "ageData":[], "doctorData":[],
               "hospitalData":[], "hospitalUnitData":[], "dateData":[], 
               "locationData":[], "numberData":[], "firstNameData":[],"phoneData":[],
              "monthData":[], "countryData":[], "genericNumberData":[] }
    
    with open(input_csv, mode='r', encoding='utf-8') as infile:
        reader = csv.DictReader(infile)  
        for i, row in enumerate(tqdm(reader)):
            phrase = row['phrase']
            secretType = row['type']
            secrets[secretType].append({ 
                'phrase': phrase
            })
            if allPhraseTypesDone(secrets, context_size):
                print("allPhraseTypesDone is done")
                break

    for key in secrets.keys():
        secrets[key] = secrets[key][0:context_size]
        
    return secrets

    
def sample_context(secretKey):
    return random.choice(context[secretKey])  #Assuming context is a global variable resulted of read_phrasePool
    
def compute_perplexity_with_context_for_secret(secretKey, secrets, checkpoint_dir, vocab_size, allowedVocab, dp=False, context_size=50):
    """Get perplexity for given examples.

    Args:
        secrets: a list, where each element is a secret dictionary
        checkpoint_dir: the directory from which to load checkpoints
        vocab_size: vocabulary size
        allowedVocab: Array of possible word that the model can output (masks the predicted distribution)
        dp: If set to True, load_gpt2_model loads DP-SGD optimizer instead of SGD.
    Returns:
        Perplexity for the input examples.
    """
    context = read_phrasePool("phrase_context_pool.csv", context_size)
    prediction_model = load_gpt2_model("gpt2", checkpoint_dir, dp)

    cce = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True, reduction=tf.keras.losses.Reduction.NONE)
    ppls = {}
    ppls_reference = []
        
    for i, secret in enumerate(secrets):
        for r in secret["datasets"]:
            d = secret["datasets"][r]
            if (d['type'] != secretKey):
                print("Secret type", secret["datasets"][r]["type"], "found, skipping")
                break
            indexed_seqs = d['data'] 
            context = sample_context(secret["datasets"][r]["type"])
            tokenized_context = tokenizer.encode(context, add_special_tokens=False)
            context_len = len(tokenized_context)
            all_losses = []
            for seq_batch in indexed_seqs:
                for seq in seq_batch:
                    full_input = tokenized_context + seq
                    tokens = tf.convert_to_tensor([full_input]) 
                    logits = prediction_model.predict(tokens).logits
                    predictions_prob = softmax(logits, axis=-1)
                    loss = 0
                    realLength = 1 #Seq has padding, repeats 2056 after the phrase ends, with RealLength we count tokens to get real phrase length
                    for k in range(context_len + 1, len(full_input)): 
                        token = full_input[k]  # Current token
                        prob = predictions_prob[0, k-1, token].numpy()  # Probability for token i given previous tokens
                        loss -= np.log2(prob)  
                        realLength += 1
                        #print("Prob of token", tokenizer.decode(token.numpy()), "is", prob)
                        if (token.numpy() == 50256):
                            break


                    all_losses.append(loss / realLength)  # Average loss over the sequence length


            print("LOSS:", all_losses)
            if r == 0:
                ppls_reference.append(all_losses)
            else:
                ppls[r] = all_losses

    return ppls, ppls_reference


## Perplexity using word probability
Computing perplexity using xi as words instead of tokens

In [ ]:
%pip install wordsprobability

In [ ]:
from tensorflow.nn import softmax
import numpy as np
import tensorflow as tf
from tqdm import tqdm
from wordsprobability import get_surprisal_per_word

def compute_perplexity_WORDLEVEL_for_secret(secretKey, secrets, checkpoint_dir, vocab_size, allowedVocab, dp=False):
    """Get perplexity for given examples.

    Args:
        secrets: a list, where each element is a secret dictionary
        checkpoint_dir: the directory from which to load checkpoints
        vocab_size: vocabulary size
        allowedVocab: Array of possible word that the model can output (masks the predicted distribution)
        dp: If set to True, load_gpt2_model loads DP-SGD optimizer instead of SGD.
    Returns:
        Perplexity for the input examples.
    """
    prediction_model = load_gpt2_model("gpt2", checkpoint_dir, dp)

    cce = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True, reduction=tf.keras.losses.Reduction.NONE)
    ppls = {}
    ppls_reference = []
    allowedTokens = []
    for vocabWord in allowedVocab:
        tokenized = tokenizer.encode(vocabWord, add_special_tokens=False) 
        allowedTokens.extend(tokenized) 
        
    allowedTokens = list(set(allowedTokens))
    for i, secret in enumerate(secrets):
        all_losses = []
        repetitions = secret["repetitions"]
        phrase = secret["phrase"]
        secretType = secret["type"]
        if (secretType != secretKey):
            print("Skipping", secretType, "while calculating", secretKey)
            continue
        surprisalDF = get_surprisal_per_word(text=phrase, model_name='gpt2-small')
        phrase_log_perplexity = sum(surprisalDF["surprisal"])
        all_losses.append(phrase_log_perplexity)
        print("PERPLEXITY:", phrase_log_perplexity)
        if repetitions == 0:
            ppls_reference.append(phrase_log_perplexity)
        else:
            ppls[repetitions] = all_losses

    return ppls, ppls_reference


# Run finetuning on GPT2 with MIMIC-III dataset.

In [16]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token


train_dataset, validation_dataset = get_dataset(
    filename= "NOTEEVENTS_deanonymized.csv",
    batch_size=BATCH_SIZE)


checkpoint_dir = 'DP-Epsilon1.5-checkpoints'
print("dataset", "loaded")
history, callbacks = train_model(train_dataset, validation_dataset, batch_size=BATCH_SIZE, vocab_size=tokenizer.vocab_size, train_epochs=TRAIN_EPOCHS, checkpoint_dir=checkpoint_dir, dp=True)

Token indices sequence length is longer than the specified maximum sequence length for this model (14511582 > 1024). Running this sequence through the model will result in indexing errors


dataset loaded
train steps 55


All PyTorch model weights were used when initializing TFGPT2LMHeadModel.

All the weights of TFGPT2LMHeadModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFGPT2LMHeadModel for predictions without further training.


Using DP-SGD with noise_multiplier = 5
Epoch 1/4



Epoch 1: loss improved from inf to 4.01283, saving model to DP-Epsilon1.5-checkpoints/ckpt_1
55/55 - 1974s - loss: 4.0128 - 1974s/epoch - 36s/step
Epoch 2/4

Epoch 2: loss improved from 4.01283 to 4.00301, saving model to DP-Epsilon1.5-checkpoints/ckpt_2
55/55 - 1921s - loss: 4.0030 - 1921s/epoch - 35s/step
Epoch 3/4

Epoch 3: loss did not improve from 4.00301
55/55 - 1925s - loss: 4.0037 - 1925s/epoch - 35s/step
Epoch 4/4

Epoch 4: loss improved from 4.00301 to 3.94374, saving model to DP-Epsilon1.5-checkpoints/ckpt_4
55/55 - 1928s - loss: 3.9437 - 1928s/epoch - 35s/step


## Experiment 1:
Evaluating exposure for GPT2 finedtuned and GP2 base

In [24]:
#1-5 Repetitions

secretsAmount = 500
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
secrets, allowedVocab = generate_text_secrets_and_references(input_csv="secrets.txt", amount_secrets=secretsAmount, amountReferences=10, maxRepetitions = 5, minRepetitions = 1)

checkpoint_dir =  'DP-Epsilon1.5-checkpoints'
print("Calculating exposure of Finetuned model with 1 to 5 repetitions")
with open("exposureResults", mode='a', encoding='utf-8') as outfile:
            writer = csv.writer(outfile)
            writer.writerow(["1 to 5 repetitions:"])


for secretKey in secrets.keys():
    if (len(secrets[secretKey]) < secretsAmount):
        print("Skipping", secretKey, "not enough repetitions:",len(secrets[secretKey]))
    print("Calculating exposure of", secretKey)
    ppls, ppls_reference = compute_perplexity_for_secret(secretKey,
      secrets[secretKey], checkpoint_dir, tokenizer.vocab_size, allowedVocab, dp=True)
    print("ppls", ppls)
    print("ppls reference", ppls_reference)
    if ppls != {}:
        exposure_interpolation = compute_exposure_interpolation(ppls, ppls_reference)
        print('  Interpolation of:', secretKey, '; '.join([f'avg_exposure={np.mean(next(iter(exposure_interpolation.values()))):.2f}±{np.std(next(iter(exposure_interpolation.values()))):.2f}' ]))
        with open("exposureResults", mode='a', encoding='utf-8') as outfile:
            writer = csv.writer(outfile)
            writer.writerow(['Using ', checkpoint_dir,' secretKey:', secretKey, np.mean(next(iter(exposure_interpolation.values())))])
"""
checkpoint_dir =  'base model'
print("Calculating exposure of base model with 1 to 5 repetitions")
for secretKey in secrets.keys(): 
    print("Calculating exposure of", secretKey)
    ppls, ppls_reference = compute_perplexity_for_secret(secretKey,
      secrets[secretKey], checkpoint_dir, tokenizer.vocab_size, allowedVocab)
    print("ppls", ppls)
    print("ppls reference", ppls_reference)
    if ppls != {}:
        exposure_interpolation = compute_exposure_interpolation(ppls, ppls_reference)
        print('  Interpolation of:', secretKey, '; '.join([f'avg_exposure={np.mean(next(iter(exposure_interpolation.values()))):.2f}±{np.std(next(iter(exposure_interpolation.values()))):.2f}' ]))
        with open("exposureResults", mode='a', encoding='utf-8') as outfile:
            writer = csv.writer(outfile)
            writer.writerow(['Using ', checkpoint_dir,' secretKey:', secretKey, np.mean(next(iter(exposure_interpolation.values())))])


#5-10 Repetitions

tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
secrets, allowedVocab = generate_text_secrets_and_references(input_csv="secrets.txt", amount_secrets=secretsAmount, amountReferences=10, maxRepetitions = 10, minRepetitions = 5)

checkpoint_dir =  'DP-Epsilon1.5-checkpoints'
print("Calculating exposure of Finetuned model with 5 to 10 repetitions", secretKey)
with open("exposureResults", mode='a', encoding='utf-8') as outfile:
            writer = csv.writer(outfile)
            writer.writerow(["5 to 10 repetitions:"])


for secretKey in secrets.keys(): 
    print("Calculating exposure of", secretKey)
    ppls, ppls_reference = compute_perplexity_for_secret(secretKey,
      secrets[secretKey], checkpoint_dir, tokenizer.vocab_size, allowedVocab, dp=True)
    print("ppls", ppls)
    print("ppls reference", ppls_reference)
    if ppls != {}:
        exposure_interpolation = compute_exposure_interpolation(ppls, ppls_reference)
        print('  Interpolation of:', secretKey, '; '.join([f'avg_exposure={np.mean(next(iter(exposure_interpolation.values()))):.2f}±{np.std(next(iter(exposure_interpolation.values()))):.2f}' ]))
        with open("exposureResults", mode='a', encoding='utf-8') as outfile:
            writer = csv.writer(outfile)
            writer.writerow(['Using ', checkpoint_dir,' secretKey:', secretKey, np.mean(next(iter(exposure_interpolation.values())))])


checkpoint_dir =  'base model'
print("Calculating exposure of base model with 5 to 10 repetitions", secretKey)
for secretKey in secrets.keys(): 
    print("Calculating exposure of", secretKey)
    ppls, ppls_reference = compute_perplexity_for_secret(secretKey,
      secrets[secretKey], checkpoint_dir, tokenizer.vocab_size, allowedVocab)
    print("ppls", ppls)
    print("ppls reference", ppls_reference)
    if ppls != {}:
        exposure_interpolation = compute_exposure_interpolation(ppls, ppls_reference)
        print('  Interpolation of:', secretKey, '; '.join([f'avg_exposure={np.mean(next(iter(exposure_interpolation.values()))):.2f}±{np.std(next(iter(exposure_interpolation.values()))):.2f}' ]))
        with open("exposureResults", mode='a', encoding='utf-8') as outfile:
            writer = csv.writer(outfile)
            writer.writerow(['Using ', checkpoint_dir,' secretKey:', secretKey, np.mean(next(iter(exposure_interpolation.values())))])
with open("exposureResults", mode='a', encoding='utf-8') as outfile:
            writer = csv.writer(outfile)
            writer.writerow(["5 to 10 repetitions:"])



#10-15 Repetitions

tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
secrets, allowedVocab = generate_text_secrets_and_references(input_csv="secrets.txt", amount_secrets=secretsAmount, amountReferences=10, maxRepetitions = 15, minRepetitions = 10)

checkpoint_dir =  'DP-Epsilon10-checkpoints'
print("Calculating exposure of Finetuned model with 10 to 15 repetitions", secretKey)
with open("exposureResults", mode='a', encoding='utf-8') as outfile:
            writer = csv.writer(outfile)
            writer.writerow(["10 to 15 repetitions:"])


for secretKey in secrets.keys(): 
    print("Calculating exposure of", secretKey)
    ppls, ppls_reference = compute_perplexity_for_secret(secretKey,
      secrets[secretKey], checkpoint_dir, tokenizer.vocab_size, allowedVocab, dp=True)
    print("ppls", ppls)
    print("ppls reference", ppls_reference)
    if ppls != {}:
        exposure_interpolation = compute_exposure_interpolation(ppls, ppls_reference)
        print('  Interpolation of:', secretKey, '; '.join([f'avg_exposure={np.mean(next(iter(exposure_interpolation.values()))):.2f}±{np.std(next(iter(exposure_interpolation.values()))):.2f}' ]))
        with open("exposureResults", mode='a', encoding='utf-8') as outfile:
            writer = csv.writer(outfile)
            writer.writerow(['Using ', checkpoint_dir,' secretKey:', secretKey, np.mean(next(iter(exposure_interpolation.values())))])

checkpoint_dir =  'base model'
print("Calculating exposure of base model with 10 to 15 repetitions")
for secretKey in secrets.keys(): 
    print("Calculating exposure of", secretKey)
    ppls, ppls_reference = compute_perplexity_for_secret(secretKey,
      secrets[secretKey], checkpoint_dir, tokenizer.vocab_size, allowedVocab)
    print("ppls", ppls)
    print("ppls reference", ppls_reference)
    if ppls != {}:
        exposure_interpolation = compute_exposure_interpolation(ppls, ppls_reference)
        print('  Interpolation of:', secretKey, '; '.join([f'avg_exposure={np.mean(next(iter(exposure_interpolation.values()))):.2f}±{np.std(next(iter(exposure_interpolation.values()))):.2f}' ]))
        with open("exposureResults", mode='a', encoding='utf-8') as outfile:
            writer = csv.writer(outfile)
            writer.writerow(['Using ', checkpoint_dir,' secretKey:', secretKey, np.mean(next(iter(exposure_interpolation.values())))])


#15-20 Repetitions

tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
secrets, allowedVocab = generate_text_secrets_and_references(input_csv="secrets.txt", amount_secrets=secretsAmount, amountReferences=10, maxRepetitions = 20, minRepetitions = 15)

checkpoint_dir =  'DP-Epsilon1.5-checkpoints'
print("Calculating exposure of Finetuned model with 15 to 20 repetitions")
with open("exposureResults", mode='a', encoding='utf-8') as outfile:
            writer = csv.writer(outfile)
            writer.writerow(["15 to 20 repetitions:"])


for secretKey in secrets.keys(): 
    print("Calculating exposure of", secretKey)
    ppls, ppls_reference = compute_perplexity_for_secret(secretKey,
      secrets[secretKey], checkpoint_dir, tokenizer.vocab_size, allowedVocab, dp=False)
    print("ppls", ppls)
    print("ppls reference", ppls_reference)
    if ppls != {}:
        exposure_interpolation = compute_exposure_interpolation(ppls, ppls_reference)
        print('  Interpolation of:', secretKey, '; '.join([f'avg_exposure={np.mean(next(iter(exposure_interpolation.values()))):.2f}±{np.std(next(iter(exposure_interpolation.values()))):.2f}' ]))
        with open("exposureResults", mode='a', encoding='utf-8') as outfile:
            writer = csv.writer(outfile)
            writer.writerow(['Using ', checkpoint_dir,' secretKey:', secretKey, np.mean(next(iter(exposure_interpolation.values())))])

checkpoint_dir =  'base model'
print("Calculating exposure of base model with 15 to 20 repetitions", secretKey)
for secretKey in secrets.keys(): 
    print("Calculating exposure of", secretKey)
    ppls, ppls_reference = compute_perplexity_for_secret(secretKey,
      secrets[secretKey], checkpoint_dir, tokenizer.vocab_size, allowedVocab)
    print("ppls", ppls)
    print("ppls reference", ppls_reference)
    if ppls != {}:
        exposure_interpolation = compute_exposure_interpolation(ppls, ppls_reference)
        print('  Interpolation of:', secretKey, '; '.join([f'avg_exposure={np.mean(next(iter(exposure_interpolation.values()))):.2f}±{np.std(next(iter(exposure_interpolation.values()))):.2f}' ]))
        with open("exposureResults", mode='a', encoding='utf-8') as outfile:
            writer = csv.writer(outfile)
            writer.writerow(['Using ', checkpoint_dir,' secretKey:', secretKey, np.mean(next(iter(exposure_interpolation.values())))]) 
"""


1/1 [==============================] - 1s 586ms/step
LOSS: [5.843111879059246]
1/1 [==============================] - 1s 553ms/step
LOSS: [5.314199632831982]
1/1 [==============================] - 1s 576ms/step
LOSS: [5.367572848285947]
1/1 [==============================] - 1s 574ms/step
LOSS: [5.780146007026945]
1/1 [==============================] - 1s 565ms/step
LOSS: [5.161243163049221]
1/1 [==============================] - 1s 545ms/step
LOSS: [5.88739994594029]
1/1 [==============================] - 1s 553ms/step
LOSS: [5.992415828364236]
1/1 [==============================] - 1s 619ms/step
LOSS: [5.782273037093026]
1/1 [==============================] - 1s 618ms/step
LOSS: [6.006551478590284]
1/1 [==============================] - 1s 613ms/step
LOSS: [6.063021683267185]
1/1 [==============================] - 1s 575ms/step
LOSS: [6.053772858210972]
1/1 [==============================] - 1s 622ms/step
LOSS: [5.556889214685985]
1/1 [==============================] - 1s 614ms/step


KeyboardInterrupt: 

In [ ]:
secretsAmount = 500
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
secrets, allowedVocab = generate_text_secrets_and_references(input_csv="secrets.txt", amount_secrets=secretsAmount, amountReferences=10, maxRepetitions = 5, minRepetitions = 1)

checkpoint_dir =  'DP-Epsilon1.5-checkpoints'
print("Calculating exposure of Finetuned model with 1 to 5 repetitions")
with open("exposureResults", mode='a', encoding='utf-8') as outfile:
            writer = csv.writer(outfile)
            writer.writerow(["1 to 5 repetitions:"])


        
print("Calculating exposure of", secretKey)
ppls, ppls_reference = compute_perplexity_for_secret("firstNameData",
secrets["firstNameData"], checkpoint_dir, tokenizer.vocab_size, allowedVocab, dp=True)
print("ppls", ppls)
print("ppls reference", ppls_reference)
if ppls != {}:
    exposure_interpolation = compute_exposure_interpolation(ppls, ppls_reference)
    print('  Interpolation of:', "firstNameData", '; '.join([f'avg_exposure={np.mean(next(iter(exposure_interpolation.values()))):.2f}±{np.std(next(iter(exposure_interpolation.values()))):.2f}' ]))
    with open("exposureResults", mode='a', encoding='utf-8') as outfile:
        writer = csv.writer(outfile)
        writer.writerow(['Using ', checkpoint_dir,' secretKey:', "firstNameData", np.mean(next(iter(exposure_interpolation.values())))])

## Experiment 2
Evaluating exposure with mask, without mask and with word level exposure


In [ ]:

secretsAmount = 50
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
secrets, allowedVocab = generate_text_secrets_and_references(input_csv="secrets.txt", amount_secrets=secretsAmount, amountReferences=10, maxRepetitions = 5, minRepetitions = 1)

checkpoint_dir =  'base model'
print("Calculating exposure of base model with word level mask, using phrases with 1 to 5 repetitions")
with open("exposureResults", mode='a', encoding='utf-8') as outfile:
            writer = csv.writer(outfile)
            writer.writerow(["Using word level"])


for secretKey in secrets.keys():
    if (len(secrets[secretKey]) < secretsAmount):
        print("Skipping", secretKey, "not enough repetitions:",len(secrets[secretKey]))
    print("Calculating exposure of", secretKey)
    ppls, ppls_reference = compute_perplexity_WORDLEVEL_for_secret(secretKey,
      secrets[secretKey], checkpoint_dir, tokenizer.vocab_size, allowedVocab)

    print("ppls", ppls)
    print("ppls reference", ppls_reference)
    if ppls != {}:
        exposure_interpolation = compute_exposure_interpolation(ppls, ppls_reference)
        print('  Interpolation of:', secretKey, '; '.join([f'avg_exposure={np.mean(next(iter(exposure_interpolation.values()))):.2f}±{np.std(next(iter(exposure_interpolation.values()))):.2f}' ]))
        with open("exposureResults", mode='a', encoding='utf-8') as outfile:
            writer = csv.writer(outfile)
            writer.writerow(['Using ', checkpoint_dir,' secretKey:', secretKey, np.mean(next(iter(exposure_interpolation.values())))])



with open("exposureResults", mode='a', encoding='utf-8') as outfile:
            writer = csv.writer(outfile)
            writer.writerow(['Using token level'])
for secretKey in secrets.keys():
    if (len(secrets[secretKey]) < secretsAmount):
        print("Skipping", secretKey, "not enough repetitions:",len(secrets[secretKey]))
    print("Calculating exposure of", secretKey)
    ppls, ppls_reference = compute_perplexity_for_secret(secretKey,
      secrets[secretKey], checkpoint_dir, tokenizer.vocab_size, allowedVocab)

    print("ppls", ppls)
    print("ppls reference", ppls_reference)
    if ppls != {}:
        exposure_interpolation = compute_exposure_interpolation(ppls, ppls_reference)
        print('  Interpolation of:', secretKey, '; '.join([f'avg_exposure={np.mean(next(iter(exposure_interpolation.values()))):.2f}±{np.std(next(iter(exposure_interpolation.values()))):.2f}' ]))
        with open("exposureResults", mode='a', encoding='utf-8') as outfile:
            writer = csv.writer(outfile)
            writer.writerow(['Using ', checkpoint_dir,' secretKey:', secretKey, np.mean(next(iter(exposure_interpolation.values())))])

## Experiment 3:

SGD model with constant prefix compared to SGD model with variant prefixes

In [ ]:
"""
secretsAmount = 50
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
secrets, allowedVocab = generate_text_secrets_and_references(input_csv="secrets.txt", amount_secrets=secretsAmount, amountReferences=10, maxRepetitions = 5, minRepetitions = 1)

checkpoint_dir =  'checkpoints'
print("Calculating exposure of SGD model with fixed prefixes")
with open("exposureResults2", mode='a', encoding='utf-8') as outfile:
            writer = csv.writer(outfile)
            writer.writerow(["Using fixed prefixes"])


for secretKey in secrets.keys():
    if (len(secrets[secretKey]) < secretsAmount):
        print("Skipping", secretKey, "not enough repetitions:",len(secrets[secretKey]))
    print("Calculating exposure of", secretKey)
    ppls, ppls_reference = compute_perplexity_for_secret(secretKey,
      secrets[secretKey], checkpoint_dir, tokenizer.vocab_size, allowedVocab)

    print("ppls", ppls)
    print("ppls reference", ppls_reference)
    if ppls != {}:
        exposure_interpolation = compute_exposure_interpolation(ppls, ppls_reference)
        print('  Interpolation of:', secretKey, '; '.join([f'avg_exposure={np.mean(next(iter(exposure_interpolation.values()))):.2f}±{np.std(next(iter(exposure_interpolation.values()))):.2f}' ]))
        with open("exposureResults2", mode='a', encoding='utf-8') as outfile:
            writer = csv.writer(outfile)
            writer.writerow(['Using ', checkpoint_dir,' secretKey:', secretKey, np.mean(next(iter(exposure_interpolation.values())))])

"""
secretsAmount = 50
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
secrets, allowedVocab = generate_text_secrets_and_references(input_csv="secrets_variablePresets.txt", amount_secrets=secretsAmount, amountReferences=10, maxRepetitions = 5, minRepetitions = 1)

checkpoint_dir =  'checkpoints_with_variants'
print("Calculating exposure of SGD model with variant prefixes")
with open("exposureResults2", mode='a', encoding='utf-8') as outfile:
            writer = csv.writer(outfile)
            writer.writerow(["Using variant prefixes"])


for secretKey in secrets.keys():
    if (len(secrets[secretKey]) < secretsAmount):
        print("Skipping", secretKey, "not enough repetitions:",len(secrets[secretKey]))
    print("Calculating exposure of", secretKey)
    ppls, ppls_reference = compute_perplexity_for_secret(secretKey,
      secrets[secretKey], checkpoint_dir, tokenizer.vocab_size, allowedVocab)

    print("ppls", ppls)
    print("ppls reference", ppls_reference)
    if ppls != {}:
        exposure_interpolation = compute_exposure_interpolation(ppls, ppls_reference)
        print('  Interpolation of:', secretKey, '; '.join([f'avg_exposure={np.mean(next(iter(exposure_interpolation.values()))):.2f}±{np.std(next(iter(exposure_interpolation.values()))):.2f}' ]))
        with open("exposureResults2", mode='a', encoding='utf-8') as outfile:
            writer = csv.writer(outfile)
            writer.writerow(['Using ', checkpoint_dir,' secretKey:', secretKey, np.mean(next(iter(exposure_interpolation.values())))])

# Evaluating accuracy of models

In [ ]:
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118


In [ ]:
from transformers import TFAutoModelForCausalLM

# Rebuild the model architecture (must match the one used in training)
model = TFAutoModelForCausalLM.from_pretrained("gpt2")
model.load_weights("checkpoints/ckpt_4")
model.save_pretrained("checkpoints_v2")

model = AutoModelForCausalLM.from_pretrained("checkpoints_v2", from_tf=True)
model.save_pretrained("checkpoints_pytorch")

tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
tokenizer.save_pretrained("checkpoints_pytorch")